# Advanced Computational Physics


## More about Python: Functions, Classes and Symbolic computing
### Object oriented python: Inheritance, exceptions and testing classes


#### *X. Cid Vidal, J.A. Hernando Morata*, in collaboration with *G. Martínez-Lema*, *M. Kekic*.
####  USC, October 2026

In [1]:
import time
print(' Last revision ', time.asctime())

 Last revision  Sat Oct  3 21:53:23 2026


## 1. Inheritance

###  1.1 Concept of inheritance

The next figure shows a tree of vehicles. All of them are vehicles, but the boat, of course, has no wheels. A bike or a car are wheeled vehicles. But a car has a motor and bikes don't.

<img src="./img/classes_inheritance.jpeg" width="500" height="400">

**Inheritance** in **Object Oriented** (OO) languages allows you to set a tree of dependences between classes.

The attributes and methods of the mother class are inherited by the daughter classes, which are called **derived classes**. A derived class can add new methods, or modify (*override*) the methods of its mother. For example, a vehicle class can have a method *run()*, and each derived class can override it to define its maximum speed, or the medium on which it runs.

Consider the matrix class. A square matrix, with the same number of rows and columns, is a specific case of a general matrix: it derives from the matrix class.

Inheritance allows you to write code at different levels. There is a basic code that runs on all kinds of vehicles, a more specific code that runs only on bicycles, and other that runs only on cars. If you have a code that runs on vehicles, and you create a new vehicle class, for example a train, the code will also run for trains! This makes the code reusable.

Inheritance is fundamental in some languages, for example C++. It is less central in Python, thanks to duck typing (see section 1.5), but we use it all the time without noticing: for example, all the exceptions are a tree of derived classes (section 3).

### 1.2 Interface and base class

There are two kinds of primordial mother classes: **interfaces** and **base classes**. The difference between both is their use.

We define an *interface* when we want to enforce that the derived classes have a given set of methods. The methods of an interface are *abstract*: they do nothing, the derived classes must implement them.

We are used to interfaces in the real world. Electricity plugs have well defined *hardware* interfaces. In Continental Europe plugs have two cylindrical pins; unfortunately the U.K. plugs are different, the pins are flat! Europe and the U.K. do not share the same interface, and we need an adapter when we travel to the U.K.!

<img src="https://images.squarespace-cdn.com/content/v1/5243143fe4b07ea3b53954f0/1389124577039-94NM388FC4QWD5Z6ZEMZ/bgui_largekit.jpg" alt="drawing" width="450"/>

A *base class* is a class expected to have some derived classes, whose methods are implemented: they provide a basic behaviour. The derived classes can change that behaviour, if they need to.

In Python, the module *abc* (abstract base classes) allows us to mix both: a class deriving from *ABC* can have *abstract methods*, marked with the decorator *@abstractmethod*, and also normal methods.

### 1.3 An example: shapes

The following *Shape* class defines the methods we expect in any shape: *perimeter()* and *area()*. They are abstract.

It also has a normal method, *describe()*, that uses them: it works for any shape, even if it does not know how to compute its area!

There are two derived classes, *Disk* and *Rectangle*, that implement these methods, and a third one, *Square*, that derives from *Rectangle*.

In [2]:
import math
from abc import ABC, abstractmethod

class Shape(ABC):
    """ base class for 2D shapes
    """

    @abstractmethod
    def perimeter(self):
        """ returns the perimeter of the shape
        """

    @abstractmethod
    def area(self):
        """ returns the area of the shape
        """

    def describe(self):
        """ returns a string with the name, the area and the perimeter
        """
        return f'{type(self).__name__}: area = {self.area():.4g}, perimeter = {self.perimeter():.4g}'


class Disk(Shape):
    """ Disk Shape class
    """

    def __init__(self, radius):
        if radius < 0:
            raise ValueError(f'the radius must be positive, not {radius}')
        self.radius = radius

    def perimeter(self):
        return 2. * math.pi * self.radius

    def area(self):
        return math.pi * self.radius**2


class Rectangle(Shape):
    """ Rectangle Shape class
    """

    def __init__(self, width, height):
        self.width  = width
        self.height = height

    def perimeter(self):
        return 2. * (self.width + self.height)

    def area(self):
        return self.width * self.height


class Square(Rectangle):
    """ Square Shape class
    """

    def __init__(self, size):
        super().__init__(size, size)

The syntax to define a derived class is to put the name of the mother class between parentheses: *class DerivedClass(MotherClass):*.

*Disk* and *Rectangle* derive from *Shape*. They have constructors and implement the abstract methods. *Disk* also checks that its radius is positive (section 3).

*Square* derives from *Rectangle*. Its constructor calls the constructor of *Rectangle*, via *super()*, which gives access to the methods of the mother class. *Square* reuses the methods *perimeter()* and *area()* of *Rectangle*: they do not need to be implemented again! And all of them inherit *describe()* from *Shape*.

In the next cell, we construct a disk, a rectangle and a square, and compute their total area. The loop does not need to know which kind of shape each object is (this is called *polymorphism*):

In [3]:
shapes = [Disk(1.), Rectangle(2., 3.), Square(2.)]

for shape in shapes:
    print(shape.describe())

print('Total area', sum(shape.area() for shape in shapes))

Disk: area = 3.142, perimeter = 6.283
Rectangle: area = 6, perimeter = 10
Square: area = 4, perimeter = 8
Total area 13.141592653589793


A square is a rectangle, and also a shape. The attribute *\_\_mro\_\_* (method resolution order) shows where Python looks for a method, from the class to its ancestors:

In [4]:
square = Square(2.)
print(isinstance(square, Rectangle), isinstance(square, Shape))
print(Square.__mro__)

True True
(<class '__main__.Square'>, <class '__main__.Rectangle'>, <class '__main__.Shape'>, <class 'abc.ABC'>, <class 'object'>)


A class with abstract methods that are not implemented cannot be instantiated. This is how the interface is enforced:

In [5]:
class Triangle(Shape):
    def __init__(self, a, b, c):
        self.sides = (a, b, c)

    def perimeter(self):
        return sum(self.sides)

try:
    Triangle(3, 4, 5)
except TypeError as error:
    print('TypeError:', error)

TypeError: Can't instantiate abstract class Triangle without an implementation for abstract method 'area'


### 1.4 Extending the builtin classes

We can also derive from the builtin classes, to extend them. In this example, we inherit from *list* and add some extra functionality:

In [6]:
class Mylist(list):

    def order(self):
        return sorted(self, reverse = True)

In [7]:
x = Mylist([1, 2, 3, 4])
print('my list  ', x, ', is of type ', type(x))

xc = x.order()
print('my order ', xc)

my list   [1, 2, 3, 4] , is of type  <class '__main__.Mylist'>
my order  [4, 3, 2, 1]


### 1.5 A note about interfaces in Python

Consider the case where *Disk* and *Rectangle* did not inherit from *Shape*, but still implemented the methods *perimeter()* and *area()*. Does the loop over the shapes still work?

Yes! For *shape.area()* to work, the only requirement is that *shape* has an *area()* method (duck typing). It does not need to derive from *Shape*.

In Python, interfaces and abstract classes are a tool to organize the code, to remind us and other developers which methods must be implemented, and to share common methods like *describe()*.

## 2. Data classes

Many classes are mostly containers of data, with a constructor that just sets the attributes. The decorator *@dataclass* writes the *\_\_init\_\_*, *\_\_repr\_\_* and *\_\_eq\_\_* methods for us, from the class attributes and their type hints:

In [8]:
from dataclasses import dataclass

@dataclass
class Particle:
    """ a particle with a name, a mass (in MeV) and a charge
    """
    name   : str
    mass   : float
    charge : int = 0

    def is_charged(self):
        return self.charge != 0

electron = Particle('electron', 0.511, -1)
print(electron)
print(electron == Particle('electron', 0.511, -1), electron.is_charged())
Particle('neutron', 939.57)

Particle(name='electron', mass=0.511, charge=-1)
True True


Particle(name='neutron', mass=939.57, charge=0)

Remember that type hints are not enforced at runtime: *Particle('electron', 'light')* does not raise any error.

*@dataclass(frozen=True)* makes the objects immutable, like *complex*: their attributes cannot be changed after construction.

## 3. Exceptions

### 3.1 Exceptions are classes

The exceptions are a tree of derived classes (see the [exception hierarchy](https://docs.python.org/3/library/exceptions.html#exception-hierarchy) in the documentation). For example, *ZeroDivisionError* derives from *ArithmeticError*, which derives from *Exception*:

In [9]:
print(ZeroDivisionError.__mro__)

(<class 'ZeroDivisionError'>, <class 'ArithmeticError'>, <class 'Exception'>, <class 'BaseException'>, <class 'object'>)


When catching exceptions, you can decide to catch the derived class or the mother class. Let's see it with the following example:

In [10]:
x, y = 1., 0.

try:
    z = x/y
except ZeroDivisionError:
    print(' Division by zero! ')

try:
    z = x/y
except ArithmeticError:
    print(' Arithmetic Error! ')

 Division by zero! 
 Arithmetic Error! 


The complete syntax is:

```python
try:
    ...                  # code that can fail
except SomeError as error:
    ...                  # executed if SomeError (or a derived class) is raised
else:
    ...                  # executed if there was no exception
finally:
    ...                  # always executed (e.g., to close a file)
```

Catch only the exceptions you expect. A bare *except:* also catches the errors in your own code, and hides them.

### 3.2 Raising exceptions

*raise* signals an error. Choose the builtin exception that describes the problem: *ValueError* for a wrong value (a negative radius), *TypeError* for a wrong type, *IndexError*, *KeyError*... and give a useful message:

In [11]:
try:
    Disk(-1.)
except ValueError as error:
    print('ValueError:', error)

ValueError: the radius must be positive, not -1.0


You can implement your own exception class, derived for example from *ZeroDivisionError*. Whoever catches *ZeroDivisionError* (or *ArithmeticError*) will also catch it:

In [12]:
class ToleranceDivisionError(ZeroDivisionError):
    """ division by a number too close to zero
    """

def divide(x, y, tol=1e-6):
    if abs(y) < tol:
        raise ToleranceDivisionError(f'|{y}| is smaller than {tol}')
    return x / y

try:
    divide(1., 1e-15)
except ZeroDivisionError as error:
    print(type(error).__name__, ':', error)

ToleranceDivisionError : |1e-15| is smaller than 1e-06


## 4. Testing classes

Let's put the shapes in a module, and write tests for them with pytest (see the notebook about testing):

In [13]:
%%writefile shapes.py
""" 2D shapes """
import math
from abc import ABC, abstractmethod

class Shape(ABC):
    """ base class for 2D shapes
    """

    @abstractmethod
    def perimeter(self):
        """ returns the perimeter of the shape
        """

    @abstractmethod
    def area(self):
        """ returns the area of the shape
        """

    def describe(self):
        """ returns a string with the name, the area and the perimeter
        """
        return f'{type(self).__name__}: area = {self.area():.4g}, perimeter = {self.perimeter():.4g}'


class Disk(Shape):
    """ Disk Shape class
    """

    def __init__(self, radius):
        if radius < 0:
            raise ValueError(f'the radius must be positive, not {radius}')
        self.radius = radius

    def perimeter(self):
        return 2. * math.pi * self.radius

    def area(self):
        return math.pi * self.radius**2


class Rectangle(Shape):
    """ Rectangle Shape class
    """

    def __init__(self, width, height):
        self.width  = width
        self.height = height

    def perimeter(self):
        return 2. * (self.width + self.height)

    def area(self):
        return self.width * self.height


class Square(Rectangle):
    """ Square Shape class
    """

    def __init__(self, size):
        super().__init__(size, size)

Writing shapes.py


In [14]:
%%writefile test_shapes.py
import math
import pytest
from shapes import Disk, Rectangle, Square

@pytest.mark.parametrize('shape, area, perimeter', [(Disk(1.), math.pi, 2 * math.pi),
                                                    (Rectangle(2., 3.), 6., 10.),
                                                    (Square(2.), 4., 8.)])
def test_known_values(shape, area, perimeter):
    assert shape.area() == pytest.approx(area)
    assert shape.perimeter() == pytest.approx(perimeter)

def test_area_scaling():
    assert Disk(3.).area() == pytest.approx(9 * Disk(1.).area())

def test_square_is_rectangle():
    assert Square(3.).area() == pytest.approx(Rectangle(3., 3.).area())

def test_negative_radius():
    with pytest.raises(ValueError):
        Disk(-1.)

Writing test_shapes.py


*pytest.raises* is used with *with*: the test passes only if the code inside the block raises a *ValueError*.

*@pytest.mark.parametrize* runs the same test for objects of different classes: it works thanks to polymorphism.

In [15]:
import sys
!{sys.executable} -m pytest -q test_shapes.py

......                                                                   [100%]
6 passed in 0.02s


## Summary

  * A derived class, *class Derived(Mother):*, inherits the attributes and methods of its mother class. It can add new methods, or override them. *super()* gives access to the methods of the mother class.

  * Abstract base classes (*abc.ABC*, *@abstractmethod*) define interfaces: the derived classes must implement the abstract methods.

  * *@dataclass* writes the constructor, *\_\_repr\_\_* and *\_\_eq\_\_* of classes that mainly hold data.

  * Exceptions are classes. Catching a mother class also catches its derived classes. Raise the builtin exception that describes the error, or derive your own.

  * *pytest.raises* checks that the code raises an exception.

### Exercises

  1. Testing the *Vector* class of the previous notebook.

      * Write it into a module *myvector.py* (with *%%writefile*), and make it raise a *ValueError* when two vectors of different dimension are added or subtracted.

      * Write a file *test_myvector.py* with tests for some algebraic properties: $u + v = v + u$, $v - v = 0$, $2v = v + v$, $|v|^2 = v \cdot v$, and the scalar product of two orthogonal vectors is zero. Use *pytest.raises* to test the *ValueError*, and *@pytest.mark.parametrize* to run the tests for several vectors. Run them with pytest.


  2. Potentials. Write an abstract base class *Potential1D*, with an abstract method *V(x)* (the potential energy), and a normal method *force(x, h=1e-5)* that computes $F = -dV/dx$ with a central difference (notebook about functions).

      * Derive the classes *Harmonic(k)*, with $V = k x^2 / 2$, and *Gravity(m, g=9.81)*, with $V = m g x$. Their constructors must raise a *ValueError* if $k \le 0$ or $m \le 0$.

      * Check the forces against the analytical results, $F = -kx$ and $F = -mg$, at a few points. Notice that you wrote *force()* only once!